# Export monthly averages + California winter rain for the website

The map files in `data/sst/` and `data/pr/` store **anomalies** (difference from the 1981–2010 average for that calendar month).
To show the **actual** ocean temperature and rainfall when you hover on the map, the site also needs those averages.

This notebook downloads 1981–2010 of `ts` and `pr` (CESM2 AMIP, `r1i1p1f1`), averages each calendar month,
coarsens to the same grid as `data/grid.json`, and writes **`data/climatology.json`** (~250 KB).

It also averages rainfall over **California** (using the state outline in `data/california.geojson`) for every wet season
(November–March) from 1950–51 to 2013–14 and writes **`data/california.csv`**, which the site uses to answer
*does El Niño bring more rain to California?*

Run it with the `elnino` kernel (Run All, a few minutes).

In [1]:
import json
import numpy as np
import pandas as pd
import xarray as xr

MODEL, MEMBER = "CESM2", "r1i1p1f1"
BASE = slice("1981-01-01", "2010-12-31")   # same base period as the anomalies
OUT = "../data"
F = 4                                       # same 4x4 coarsening as elnino_data_prep

cat = pd.read_csv("https://storage.googleapis.com/cmip6/pangeo-cmip6.csv")

def open_var(variable, table="Amon"):
    rows = cat.query(
        "source_id == @MODEL and experiment_id == 'amip' and member_id == @MEMBER "
        "and table_id == @table and variable_id == @variable"
    )
    path = rows.sort_values("version").iloc[-1].zstore
    print(path)
    return xr.open_zarr(path, consolidated=True, storage_options={"token": "anon"})[variable]

ts = open_var("ts").sel(time=BASE).load()
pr_all = open_var("pr").load()          # all years: needed for the California series
pr = pr_all.sel(time=BASE)
sftlf = open_var("sftlf", table="fx").load()
print(ts.sizes)

gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/Amon/ts/gn/v20190218/


/Users/justinjames/miniforge3/envs/elnino/lib/python3.12/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.82.2. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(
/Users/justinjames/miniforge3/envs/elnino/lib/python3.12/site-packages/google/auth/transport/grpc.py:43: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details

gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/Amon/pr/gn/v20190319/
gs://cmip6/CMIP6/CMIP/NCAR/CESM2/amip/r1i1p1f1/fx/sftlf/gn/v20190218/
Frozen({'time': 360, 'lat': 192, 'lon': 288})


In [2]:
sftlf = sftlf.assign_coords(lat=ts.lat, lon=ts.lon)
pr = pr.assign_coords(lat=ts.lat, lon=ts.lon)

sst_clim = (ts - 273.15).where(sftlf < 50).groupby("time.month").mean("time")   # °C, ocean only
pr_clim = (pr * 86400).groupby("time.month").mean("time")                         # mm/day

def to_web(da):
    da = da.coarsen(lat=F, lon=F, boundary="trim").mean()
    da = da.assign_coords(lon=((da.lon + 180) % 360) - 180).sortby("lon")
    return da

grid = json.load(open(f"{OUT}/grid.json"))
sst_w, pr_w = to_web(sst_clim), to_web(pr_clim)
assert np.allclose(sst_w.lat.values, grid["lat"], atol=0.01), "latitudes don't match grid.json"
assert np.allclose(sst_w.lon.values, grid["lon"], atol=0.01), "longitudes don't match grid.json"

def ints(a):
    a = np.round(np.asarray(a, dtype=float) * 10)
    return [None if np.isnan(v) else int(v) for v in a.ravel()]

out = {
    "base": "1981-2010",
    "scale": 0.1,
    "sst": [ints(sst_w.sel(month=m).values) for m in range(1, 13)],
    "pr": [ints(pr_w.sel(month=m).values) for m in range(1, 13)],
}
json.dump(out, open(f"{OUT}/climatology.json", "w"), separators=(",", ":"))

# sanity check: the Niño 3.4 box should be ~26–28 °C
box = sst_clim.sel(lat=slice(-5, 5), lon=slice(190, 240)).mean(("lat", "lon"))
print("Niño 3.4 average SST by month (°C):", np.round(box.values, 1))
print("wrote", f"{OUT}/climatology.json")

Niño 3.4 average SST by month (°C): [26.5 26.7 27.2 27.8 27.8 27.6 27.2 26.8 26.7 26.7 26.6 26.5]
wrote ../data/climatology.json


## California winter rainfall

* **Region:** model grid cells whose center falls inside California's state outline, averaged with area weights.
* **Winter:** the November–March wet season, labeled by its two years (e.g. 1997–98). Only complete winters are kept.
* **% of normal:** that winter's rainfall divided by the average winter of 1981–82 to 2009–10 (same base period as everything else).
* **Winter ONI:** the December–February average ONI from `data/nino34.csv`; the winter's phase is January's El Niño / La Niña label.

In [3]:
from matplotlib.path import Path as MplPath

ca = json.load(open(f"{OUT}/california.geojson"))["features"][0]["geometry"]
polys = ca["coordinates"] if ca["type"] == "MultiPolygon" else [ca["coordinates"]]
paths = [MplPath(np.array(p[0])) for p in polys]          # outer ring of each piece

lon180 = ((pr_all.lon.values + 180) % 360) - 180
LON, LAT = np.meshgrid(lon180, pr_all.lat.values)
pts = np.column_stack([LON.ravel(), LAT.ravel()])
inside = np.zeros(len(pts), dtype=bool)
for pth in paths:
    inside |= pth.contains_points(pts)
mask = xr.DataArray(inside.reshape(LAT.shape), dims=("lat", "lon"),
                    coords={"lat": pr_all.lat, "lon": pr_all.lon})
print("grid cells inside California:", int(mask.sum()))

ca_pr = (pr_all * 86400).where(mask)
ca_pr = ca_pr.weighted(np.cos(np.deg2rad(ca_pr.lat))).mean(("lat", "lon"))   # mm/day, one value per month

monthly = pd.DataFrame({
    "year": [t.year for t in ca_pr.time.values],
    "month": [t.month for t in ca_pr.time.values],
    "rain": ca_pr.values,
})
monthly["days"] = monthly.month.map({11: 30, 12: 31, 1: 31, 2: 28, 3: 31})   # CESM2 uses a no-leap calendar
wet = monthly[monthly.month.isin([11, 12, 1, 2, 3])].copy()
wet["winter"] = np.where(wet.month >= 11, wet.year + 1, wet.year)          # label by the January year
g = wet.groupby("winter")
win = pd.DataFrame({
    "n": g.size(),
    "rain_mm": g.apply(lambda d: (d.rain * d.days).sum()),               # total mm over Nov–Mar
})
win = win[win.n == 5].drop(columns="n")

base = win.loc[1982:2010, "rain_mm"].mean()                                # winters 1981–82 .. 2009–10
win["pct_normal"] = (100 * win.rain_mm / base).round(1)
win["inches"] = (win.rain_mm / 25.4).round(2)
win["normal_inches"] = round(base / 25.4, 2)

nino = pd.read_csv(f"{OUT}/nino34.csv")
djf = nino[nino.month.isin([12, 1, 2])].copy()
djf["winter"] = np.where(djf.month == 12, djf.year + 1, djf.year)
win["oni"] = djf.groupby("winter").oni.mean().round(2)
win["phase"] = nino[nino.month == 1].set_index("year").phase

win = win.reset_index()
win["label"] = (win.winter - 1).astype(str) + "–" + win.winter.astype(str).str[-2:]
win = win[["label", "winter", "oni", "phase", "pct_normal", "inches", "normal_inches"]].dropna()
win.to_csv(f"{OUT}/california.csv", index=False)

print(f"normal Nov–Mar rain: {base / 25.4:.1f} inches")
print("correlation between winter ONI and California % of normal:", round(win.oni.corr(win.pct_normal), 2))
win.sort_values("oni", ascending=False).head(8)

grid cells inside California: 32
normal Nov–Mar rain: 18.1 inches
correlation between winter ONI and California % of normal: 0.27


,label,winter,oni,phase,pct_normal,inches,normal_inches
32,1982–83,1983,2.53,El Niño,134.6,24.42,18.14
47,1997–98,1998,2.41,El Niño,182.6,33.12,18.14
41,1991–92,1992,1.77,El Niño,92.7,16.82,18.14
22,1972–73,1973,1.54,El Niño,114.8,20.82,18.14
59,2009–10,2010,1.47,El Niño,92.0,16.69,18.14
7,1957–58,1958,1.31,El Niño,83.7,15.18,18.14
36,1986–87,1987,1.26,El Niño,118.5,21.49,18.14
52,2002–03,2003,1.16,El Niño,124.1,22.51,18.14
